# Test: lectura de llegenda (pipeline real del backend)

Aquest notebook reprodueix exactament la lògica de `POST /api/legend` del backend
(`app/services/dxf_service.py` i `app/services/legend_text_service.py`),
aplicada a un fitxer `legend.dxf` de Google Drive:

1. Extreu directament del DXF tot el `TEXT`/`MTEXT` real (keys en negreta i
   labels) i tota la geometria mesurable `LINE`/`LWPOLYLINE` amb el seu estil
   `(colorHex, linetype, lineweight)` -- incloent-hi el que està niat dins
   d'`INSERT` -- sense retallar ni renderitzar cap imatge del modelspace.
2. Emparella cada mostra (swatch) amb la seva key més propera, i cada key
   amb el seu label més proper, per posició -- mai s'inventen colors,
   estils ni text: tot ve directament de la geometria i el text reals del
   DXF, sense IA ni crida de xarxa.
3. Deriva `isDashed` (booleà) de cada `linetype` -- `False` només per
   `CONTINUOUS`, `True` per qualsevol altre -- perquè el frontend pugui
   pintar el swatch de cada fila com a barra sòlida o discontínua sense
   necessitat de conèixer el `linetype` DXF real.

El resultat final (`entries`) és exactament el JSON que la resposta de
`/api/legend` retornaria. Copia'l per fer-lo servir com a input al notebook
`Test-Plan-Matching.ipynb`.

In [ ]:
# %% [Cel·la 1] Instal·lar dependències
!pip install -q ezdxf pandas

In [14]:
# %% [Cel·la 2] Muntar Google Drive
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# %% [Cel·la 3] Ruta al fitxer CleanLegend.dxf a Drive
DXF_PATH = "/content/drive/MyDrive/Colab Notebooks/plan-count/CleanLegend.dxf"
print(DXF_PATH)

/content/drive/MyDrive/Colab Notebooks/plan-count/Legend.dxf


In [16]:
# %% [Cel·la 4] Obrir el fitxer
import ezdxf

doc = ezdxf.readfile(DXF_PATH)
msp = doc.modelspace()
print("DXF llegit correctament. Entitats a l'espai model:", len(msp))

DXF llegit correctament. Entitats a l'espai model: 510


In [17]:
# %% [Cel·la 5] Còpia exacta de app/services/dxf_service.py (les funcions
# rellevants per agrupar geometria per estil, incloent-hi el mateix
# tractament de BYLAYER/BYBLOCK i la recursió dins d'INSERT que fa servir
# el backend real).
import ezdxf.colors as ezcolors
import ezdxf.lldxf.const

StyleKey = tuple


def _safe_layer(doc, name):
    try:
        return doc.layers.get(name)
    except ezdxf.lldxf.const.DXFTableEntryError:
        return None


def iter_with_blocks(entities, depth=0, max_depth=3):
    for entity in entities:
        yield entity
        if entity.dxftype() == "INSERT" and depth < max_depth:
            try:
                nested = list(entity.virtual_entities())
            except Exception:
                nested = []
            yield from iter_with_blocks(nested, depth + 1, max_depth)


def _effective_color_hex(entity, doc) -> str:
    true_color = entity.dxf.get("true_color", None)
    if true_color is not None:
        r, g, b = ezcolors.int2rgb(true_color)
        return "#%02x%02x%02x" % (r, g, b)
    aci = entity.dxf.color
    if aci == 256:
        layer = _safe_layer(doc, entity.dxf.layer)
        aci = layer.color if layer else 7
    elif aci == 0:
        aci = 7
    return "#%02x%02x%02x" % ezcolors.aci2rgb(aci)


def _effective_linetype(entity, doc) -> str:
    linetype = entity.dxf.linetype
    if linetype == "BYLAYER":
        layer = _safe_layer(doc, entity.dxf.layer)
        linetype = layer.dxf.linetype if layer else "CONTINUOUS"
    return linetype.upper()


def _effective_lineweight(entity, doc) -> int:
    lineweight = entity.dxf.lineweight
    if lineweight == -1:
        layer = _safe_layer(doc, entity.dxf.layer)
        return layer.dxf.lineweight if layer else -3
    return lineweight


def _entity_length(entity):
    dxftype = entity.dxftype()
    if dxftype == "LINE":
        s, e = entity.dxf.start, entity.dxf.end
        return ((s[0] - e[0]) ** 2 + (s[1] - e[1]) ** 2) ** 0.5
    if dxftype == "LWPOLYLINE":
        points = list(entity.get_points("xy"))
        if len(points) < 2:
            return None
        total = 0.0
        for i in range(len(points) - 1):
            a, b = points[i], points[i + 1]
            total += ((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2) ** 0.5
        if entity.closed:
            a, b = points[-1], points[0]
            total += ((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2) ** 0.5
        return total
    return None


def group_measurable_geometry_by_style(doc):
    msp = doc.modelspace()
    totals = {}
    for entity in iter_with_blocks(msp):
        length = _entity_length(entity)
        if length is None:
            continue
        style = (
            _effective_color_hex(entity, doc),
            _effective_linetype(entity, doc),
            _effective_lineweight(entity, doc),
        )
        totals[style] = totals.get(style, 0.0) + length
    return {s: t for s, t in totals.items() if t != 0.0}

In [ ]:
# %% [Cel·la 5b] Còpia exacta de dxf_service.py:iter_measurable_styles_with_position --
# com group_measurable_geometry_by_style, però sense agregar: cada entitat
# mesurable es queda amb la seva pròpia posició, necessària per emparellar
# cada mostra de línia/color amb el seu key/label més proper.
def _entity_start_point(entity):
    if entity.dxftype() == "LINE":
        start = entity.dxf.start
        return (start[0], start[1])
    first_point = next(iter(entity.get_points("xy")))
    return (first_point[0], first_point[1])


def iter_measurable_styles_with_position(doc):
    results = []
    for entity in iter_with_blocks(doc.modelspace()):
        length = _entity_length(entity)
        if length is None or length == 0.0:
            continue
        style = (
            _effective_color_hex(entity, doc),
            _effective_linetype(entity, doc),
            _effective_lineweight(entity, doc),
        )
        results.append((style, _entity_start_point(entity)))
    return results

In [ ]:
# %% [Cel·la 8] Còpia exacta de app/services/legend_text_service.py --
# extreu key/label/estil directament de la geometria i el text real del
# DXF, sense IA ni renderitzat d'imatge (substitueix per complet l'antiga
# crida a OpenRouter -- ja no calen ni OPENROUTER_API_KEY ni MODEL).
import re
from dataclasses import dataclass

_KEY_PATTERN = re.compile(r"^[A-Za-zÀ-ÿ]{0,3}\d{1,3}\*?$")
_MAX_KEY_LENGTH = 6
_BOLD_MARKER = re.compile(r"\|b1\|")


def is_dashed_linetype(linetype: str) -> bool:
    # Còpia exacta de app/services/dxf_service.py:is_dashed_linetype --
    # qualsevol linetype que no sigui CONTINUOUS es tracta com a
    # discontinu, perquè el frontend només necessita una distinció binària
    # sòlid/discontinu per pintar el swatch de la llegenda.
    return linetype.upper() != "CONTINUOUS"


@dataclass(frozen=True)
class _TextCandidate:
    text: str
    position: tuple
    is_key: bool


def _is_bold_mtext(entity) -> bool:
    return bool(_BOLD_MARKER.search(entity.dxf.text))


def _matches_key_pattern(text: str) -> bool:
    return len(text) <= _MAX_KEY_LENGTH and bool(_KEY_PATTERN.match(text))


def _extract_text_candidates(doc):
    # Recorre també el que està niat dins d'INSERT (com fa
    # iter_measurable_styles_with_position amb la geometria) -- si el key/
    # label d'una fila viu dins del mateix bloc que la seva mostra de
    # línia/color, cal trobar-lo igual.
    candidates = []
    for entity in iter_with_blocks(doc.modelspace()):
        if entity.dxftype() not in ("TEXT", "MTEXT"):
            continue
        if entity.dxftype() == "MTEXT":
            text = entity.plain_text().strip()
            is_bold_capable, is_bold = True, _is_bold_mtext(entity)
            position = (entity.dxf.insert[0], entity.dxf.insert[1])
        else:
            text = entity.dxf.text.strip()
            is_bold_capable, is_bold = False, False
            # get_placement() (no dxf.insert) és la posició real quan el
            # TEXT està justificat (halign/valign != per defecte) -- amb
            # justificació, insert sovint queda a l'origen.
            placement_point = entity.get_placement()[1]
            position = (placement_point[0], placement_point[1])
        if not text:
            continue
        is_key = _matches_key_pattern(text) and (is_bold or not is_bold_capable)
        candidates.append(_TextCandidate(text=text, position=position, is_key=is_key))
    return candidates


def _truncate_label(raw_text):
    colon_index, newline_index = raw_text.find(":"), raw_text.find("\n")
    cut_points = [i for i in (colon_index, newline_index) if i != -1]
    cut = min(cut_points) if cut_points else len(raw_text)
    return raw_text[:cut].strip()


def _distance(a, b):
    return ((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2) ** 0.5


def _median_nearest_key_distance(keys):
    distances = []
    for i, key in enumerate(keys):
        others = [_distance(key.position, other.position) for j, other in enumerate(keys) if j != i]
        if others:
            distances.append(min(others))
    if not distances:
        return 1.0
    distances.sort()
    mid = len(distances) // 2
    if len(distances) % 2 == 1:
        return distances[mid]
    return (distances[mid - 1] + distances[mid]) / 2


def _match_threshold(keys):
    return 2 * _median_nearest_key_distance(keys) if len(keys) >= 2 else 1.0


def _nearest_within(position, candidates, threshold):
    best, best_distance = None, threshold
    for candidate in candidates:
        distance = _distance(position, candidate.position)
        if distance <= best_distance:
            best, best_distance = candidate, distance
    return best


def extract_legend_entries(doc):
    text_candidates = _extract_text_candidates(doc)
    keys = [c for c in text_candidates if c.is_key]
    labels = [c for c in text_candidates if not c.is_key]
    threshold = _match_threshold(keys)

    matches = []
    for style, position in iter_measurable_styles_with_position(doc):
        key_candidate = _nearest_within(position, keys, threshold)
        if key_candidate is not None:
            matches.append((key_candidate, style))

    # Deduplica files idèntiques (mateixa key + mateix estil) -- p.ex. dues
    # entitats mesurables diferents que acaben resolent-se a la mateixa
    # combinació no han de generar dues files repetides.
    deduped_matches = []
    seen = set()
    for key_candidate, style in matches:
        dedup_key = (key_candidate.text, style)
        if dedup_key in seen:
            continue
        seen.add(dedup_key)
        deduped_matches.append((key_candidate, style))
    matches = deduped_matches

    label_by_key = {}
    for key_candidate in {key for key, _ in matches}:
        label_candidate = _nearest_within(key_candidate.position, labels, threshold)
        label_by_key[key_candidate.text] = _truncate_label(label_candidate.text) if label_candidate else ""
    for key_text in list(label_by_key):
        if key_text.endswith("*"):
            base_key = key_text.rstrip("*")
            if base_key in label_by_key:
                label_by_key[key_text] = label_by_key[base_key]

    return [
        {
            "key": key_candidate.text,
            "label": label_by_key[key_candidate.text],
            "colorHex": style[0],
            "linetype": style[1],
            "lineweight": style[2],
            "isDashed": is_dashed_linetype(style[1]),
        }
        for key_candidate, style in matches
    ]


entries = extract_legend_entries(doc)
print(f"Files de llegenda llegides: {len(entries)}")

In [ ]:
# %% [Cel·la 9] Mostrar el resultat en taula i com a JSON -- aquest JSON
# és exactament el que caldrà enganxar com a `LEGEND_JSON` al notebook
# Test-Plan-Matching.ipynb.
import json

import pandas as pd

legend_df = pd.DataFrame(entries, columns=["key", "label", "colorHex", "linetype", "lineweight", "isDashed"])
display(legend_df)

print("\n--- Copia el JSON de sota per al notebook Test-Plan-Matching.ipynb ---\n")
print(json.dumps(entries, ensure_ascii=False, indent=2))